In [1]:
import pandas as pd
import re
from collections import Counter
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
# ¡NUEVAS IMPORTACIONES!
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score, accuracy_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE ARCHIVOS ---
# 1. Rutas a los 4 archivos raw de selección
archivos_raw = [
    r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\Python\variables_finales_dm_j48",
    r"C:\Users\benja\Desktop\BD PAMPA\Calculos Fingerprint\J48\variables_finales_j48",
    r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\Python\variables_finales_dm_ibk",
    r"C:\Users\benja\Desktop\BD PAMPA\Calculos Fingerprint\IBK\variables_finales_ibk"
]

# 2. Rutas a los archivos de datos completos
ruta_entrenamiento_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/combined_training.csv"
ruta_prueba_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_test.csv"
ruta_externa_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_external.csv"

print("✅ Celda 1: Librerías y configuración cargadas.")

✅ Celda 1: Librerías y configuración cargadas.


In [3]:
# --- PARÁMETROS DEL FILTRO DE FRECUENCIA ---
# ¡AJUSTA ESTOS VALORES SEGÚN TU NECESIDAD!
FRECUENCIA_MINIMA = 3
FRECUENCIA_MAXIMA = 4
# ---------------------------------------------

print("--- Iniciando Fase de Consenso ---")

# 1. Unir los 4 archivos raw y contar frecuencias
todos_los_indices = []
for ruta_raw in archivos_raw:
    with open(ruta_raw, 'r') as f:
        for linea in f:
            try:
                parte_util = linea.split('[', 1)[1].rsplit(']', 1)[0]
                numeros = re.findall(r'\d+', parte_util)
                todos_los_indices.extend([int(num) for num in numeros])
            except IndexError:
                continue

frecuencia_indices = Counter(todos_los_indices)

# 2. Filtrar por rango de frecuencia y traducir a nombres
df_headers = pd.read_csv(ruta_entrenamiento_completo, sep=',', nrows=0)
lista_nombres_atributos = df_headers.columns.tolist()
columna_clase = df_headers.columns[-1]

atributos_consenso = []
for indice, frecuencia in frecuencia_indices.items():
    if FRECUENCIA_MINIMA <= frecuencia <= FRECUENCIA_MAXIMA and 0 < indice <= len(lista_nombres_atributos):
        nombre_atributo = lista_nombres_atributos[indice - 1]
        if nombre_atributo != 'LOGPcons':
            atributos_consenso.append(nombre_atributo)

if not atributos_consenso:
    print(f"\n❌ ¡ALERTA! No se encontraron atributos con frecuencia entre {FRECUENCIA_MINIMA} y {FRECUENCIA_MAXIMA}. Prueba un rango diferente.")
else:
    print(f"\n✅ Celda 2: Se encontraron {len(atributos_consenso)} atributos de consenso.")

--- Iniciando Fase de Consenso ---

✅ Celda 2: Se encontraron 71 atributos de consenso.


In [4]:
print("--- Creando Datasets de Consenso... ---")

if not atributos_consenso:
    print("❌ No se pueden crear los datasets porque no se encontraron atributos de consenso.")
else:
    columnas_finales = sorted(atributos_consenso) + [columna_clase]
    
    # Crear y guardar los 3 archivos CSV
    df_train = pd.read_csv(ruta_entrenamiento_completo)[columnas_finales]
    df_train.to_csv("consenso_train.csv", index=False)

    df_test = pd.read_csv(ruta_prueba_completo)[columnas_finales]
    df_test.to_csv("consenso_test.csv", index=False)

    df_external = pd.read_csv(ruta_externa_completo)[columnas_finales]
    df_external.to_csv("consenso_external.csv", index=False)
    
    print("✅ Celda 3: Datasets de consenso creados con éxito.")
    
    # Preparamos los datos para la siguiente celda
    X_train, y_train = df_train.drop(columns=[columna_clase]), df_train[columna_clase]
    X_test, y_test = df_test.drop(columns=[columna_clase]), df_test[columna_clase]
    X_external, y_external = df_external.drop(columns=[columna_clase]), df_external[columna_clase]

--- Creando Datasets de Consenso... ---
✅ Celda 3: Datasets de consenso creados con éxito.


In [5]:
print("--- Iniciando Evaluación Comparativa de Modelos... ---")

modelos_a_evaluar = {
    "SVM": SVC(probability=True, random_state=42), 
    "Random Forest": RandomForestClassifier(random_state=42),
    "Árbol de Decisión (J48)": DecisionTreeClassifier(random_state=42),
    "k-NN (k=5)": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

resultados_comparativos = {}
for nombre_modelo, modelo in modelos_a_evaluar.items():
    modelo.fit(X_train, y_train)
    roc_auc_test = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
    resultados_comparativos[nombre_modelo] = roc_auc_test
    print(f"  > {nombre_modelo:<25} | ROC AUC (Test): {roc_auc_test:.4f}")

campeon_nombre = max(resultados_comparativos, key=resultados_comparativos.get)
print(f"\n✅ Celda 4: El modelo campeón en la comparativa es: {campeon_nombre}")

--- Iniciando Evaluación Comparativa de Modelos... ---
  > SVM                       | ROC AUC (Test): 0.7676
  > Random Forest             | ROC AUC (Test): 0.8075
  > Árbol de Decisión (J48)   | ROC AUC (Test): 0.6567
  > k-NN (k=5)                | ROC AUC (Test): 0.7496
  > Naive Bayes               | ROC AUC (Test): 0.7689

✅ Celda 4: El modelo campeón en la comparativa es: Random Forest


In [6]:
print(f"--- Optimizando al Campeón ({campeon_nombre})... ---")

# Grids de parámetros para los modelos más prometedores
param_grids = {
    "Random Forest": {'n_estimators': [100, 200, 300], 'max_features': ['sqrt', 'log2']},
    "SVM": {'C': [1, 10, 100], 'gamma': ['scale', 'auto']}
}

if campeon_nombre in param_grids:
    grid_search = GridSearchCV(modelos_a_evaluar[campeon_nombre], param_grids[campeon_nombre], cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)
    modelo_final = grid_search.best_estimator_
    print(f"\n> Mejores parámetros encontrados: {grid_search.best_params_}")
else:
    print(f"> No hay grid de optimización definido para {campeon_nombre}. Se usará el modelo base.")
    modelo_final = modelos_a_evaluar[campeon_nombre]

# --- REPORTE FINAL ACTUALIZADO ---
print("\n" + "="*60 + "\n--- REPORTE FINAL DEL MODELO DE CONSENSO ---\n" + "="*60)
for nombre_set, X_eval, y_eval in [("Prueba Interna (Test Set)", X_test, y_test), ("Prueba Externa (External Set)", X_external, y_external)]:
    y_pred = modelo_final.predict(X_eval)
    y_proba = modelo_final.predict_proba(X_eval)[:, 1]
    
    print(f"\nResultados en: {nombre_set}")
    print("-" * 40)
    # --- ¡MÉTRICAS NUEVAS Y ANTERIORES! ---
    print(f"  Accuracy:         {accuracy_score(y_eval, y_pred):.4f}")
    print(f"  ROC AUC:          {roc_auc_score(y_eval, y_proba):.4f}")
    print(f"  BACC:             {balanced_accuracy_score(y_eval, y_pred):.4f}")
    print(f"  Sensitivity:      {recall_score(y_eval, y_pred, pos_label='Act1'):.4f}")
    print(f"  Specificity:      {recall_score(y_eval, y_pred, pos_label='Act-1'):.4f}")
    print(f"  Kappa:            {cohen_kappa_score(y_eval, y_pred):.4f}")

print("\n\n✅ ¡PROYECTO COMPLETADO!")

--- Optimizando al Campeón (Random Forest)... ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits

> Mejores parámetros encontrados: {'max_features': 'sqrt', 'n_estimators': 300}

--- REPORTE FINAL DEL MODELO DE CONSENSO ---

Resultados en: Prueba Interna (Test Set)
----------------------------------------
  Accuracy:         0.7422
  ROC AUC:          0.8130
  BACC:             0.7389
  Sensitivity:      0.7856
  Specificity:      0.6923
  Kappa:            0.4798

Resultados en: Prueba Externa (External Set)
----------------------------------------
  Accuracy:         0.8848
  ROC AUC:          0.6218
  BACC:             0.6218
  Sensitivity:      0.9413
  Specificity:      0.3023
  Kappa:            0.2543


✅ ¡PROYECTO COMPLETADO!
